In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

In [3]:
countries = pd.read_csv("CountriesSD.csv")
summer = pd.read_csv("SummerSD.csv")
winter = pd.read_csv("WinterSD.csv")

In [4]:
clean_countries = countries.dropna()
clean_summer = summer.dropna()
clean_winter = winter.dropna()

In [40]:
clean_winter.loc["Season"] = "Winter"
clean_summer.loc["Season"] = "Summer"

/var/folders/1f/gq9hvw455wjg13c6mbjdd5dw0000gn/T/ipykernel_53159/2784069801.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_summer.loc["Season"] = "Summer"


In [10]:
both_olympics = pd.concat([clean_winter, clean_summer], ignore_index=True)
both_olympics.head()

,Unnamed: 0,Year,City,Sport,Discipline,Athlete,Country,Gender,Event,Medal,Season,Code
0,0,1924,Chamonix,Biathlon,Biathlon,"BERTHET, G.",FRA,Men,Military Patrol,Bronze,Winter,NaN
1,1,1924,Chamonix,Biathlon,Biathlon,"MANDRILLON, C.",FRA,Men,Military Patrol,Bronze,Winter,NaN
2,2,1924,Chamonix,Biathlon,Biathlon,"MANDRILLON, Maurice",FRA,Men,Military Patrol,Bronze,Winter,NaN
3,3,1924,Chamonix,Biathlon,Biathlon,"VANDELLE, André",FRA,Men,Military Patrol,Bronze,Winter,NaN
4,4,1924,Chamonix,Biathlon,Biathlon,"AUFDENBLATTEN, Adolf",SUI,Men,Military Patrol,Gold,Winter,NaN


In [11]:
both_olympics = both_olympics.drop(columns=["Unnamed: 0", "Code"])

In [12]:
both_olympics.head(3)

,Year,City,Sport,Discipline,Athlete,Country,Gender,Event,Medal,Season
0,1924,Chamonix,Biathlon,Biathlon,"BERTHET, G.",FRA,Men,Military Patrol,Bronze,Winter
1,1924,Chamonix,Biathlon,Biathlon,"MANDRILLON, C.",FRA,Men,Military Patrol,Bronze,Winter
2,1924,Chamonix,Biathlon,Biathlon,"MANDRILLON, Maurice",FRA,Men,Military Patrol,Bronze,Winter


In [13]:
all_data = both_olympics.merge(
    clean_countries,
    left_on="Country",  
    right_on="Code",
    how="left"
)

In [14]:
all_data.head(3)

,Year,City,Sport,Discipline,Athlete,Country_x,Gender,Event,Medal,Season,Unnamed: 0,Country_y,Code,Population,GDP per Capita
0,1924,Chamonix,Biathlon,Biathlon,"BERTHET, G.",FRA,Men,Military Patrol,Bronze,Winter,66.0,France,FRA,66808385.0,36205.568102
1,1924,Chamonix,Biathlon,Biathlon,"MANDRILLON, C.",FRA,Men,Military Patrol,Bronze,Winter,66.0,France,FRA,66808385.0,36205.568102
2,1924,Chamonix,Biathlon,Biathlon,"MANDRILLON, Maurice",FRA,Men,Military Patrol,Bronze,Winter,66.0,France,FRA,66808385.0,36205.568102


In [18]:
country_season_medals = all_data.groupby(["Country_y", "Season"]).size().reset_index(name="Total_Medals")

In [21]:
country_season_medals = country_season_medals.rename(columns={"Country_y": "Country"})

In [22]:
country_season_medals.head(3)

,Country,Season,Total_Medals
0,Australia,Winter,15
1,Austria,Winter,280
2,Belarus,Winter,15


In [26]:
country_info = all_data[["Country_y", "GDP per Capita", "Population"]].drop_duplicates("Country_y")

regression_data = country_season_medals.merge(
    country_info,
    left_on="Country",
    right_on="Country_y"
)

In [27]:
regression_data.head(3)

,Country,Season,Total_Medals,Country_y,GDP per Capita,Population
0,Australia,Winter,15,Australia,56310.962993,23781169.0
1,Austria,Winter,280,Austria,43774.985174,8611088.0
2,Belarus,Winter,15,Belarus,5740.456495,9513000.0


In [41]:
regression_data.isna()

,Country,Season,Total_Medals,Country_y,GDP per Capita,Population,Prop_Female
0,False,True,False,False,False,False,False
1,False,True,False,False,False,False,False
2,False,True,False,False,False,False,False
3,False,True,False,False,False,False,False
4,False,True,False,False,False,False,False
5,False,True,False,False,False,False,False
6,False,True,False,False,False,False,False
7,False,True,False,False,False,False,False
8,False,True,False,False,False,False,False
9,False,True,False,False,False,False,False


In [34]:
women_prop = all_data.groupby("Country_y")["Gender"].apply(
    lambda x: (x == "Women").mean()
).reset_index(name="Prop_Female")

In [36]:
regression_data = regression_data.merge(women_prop, on="Country_y")

In [38]:
regression_data.head(3)

,Country,Season,Total_Medals,Country_y,GDP per Capita,Population,Prop_Female
0,Australia,NaN,15,Australia,56310.962993,23781169.0,0.466667
1,Austria,NaN,280,Austria,43774.985174,8611088.0,0.267857
2,Belarus,NaN,15,Belarus,5740.456495,9513000.0,0.466667


In [39]:
regression_data["Season"] = regression_data["Season"].map({"Winter":0, "Summer":1})

X = regression_data[["GDP per Capita", "Population", "Prop_Female"]]
Y = regression_data[["Total_Medals"]]

X = sm.add_constant(X)

model = sm.GLM(Y, X, family=sm.families.Poisson()).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           Total_Medals   No. Observations:                   35
Model:                            GLM   Df Residuals:                       31
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3020.1
Date:                Thu, 19 Mar 2026   Deviance:                       5849.9
Time:                        13:47:40   Pearson chi2:                 6.05e+03
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              4.6730      0.051     91.